In [70]:
import tensorflow as tf
import numpy as np
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv.xls")
display(df.head(2))

df = df.drop(columns=['PassengerId','Name'])

df.isna().sum()

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Cabin'] = df['Cabin'].fillna(df['Cabin'].mode()[0])


df['Ticket'] = df['Ticket'].str.extract(r'^([A-Za-z./]+)')[0].fillna('numeric')
df['Cabin'] = df['Cabin'].str[0]


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [71]:
x = df.drop(columns=['Survived'])
y = df['Survived']

In [72]:
numeric_columns = x.select_dtypes(include='number').columns
categorical_columns = x.select_dtypes(include='object').columns

In [73]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [74]:
x_encoded = pd.get_dummies(x, columns=categorical_columns,drop_first=True)
x_encoded = x_encoded.astype(int)

In [75]:
x_train,x_test,y_train,y_test = train_test_split(x_encoded,y,test_size=0.2,random_state=42)

In [76]:
sc = StandardScaler()

x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.fit_transform(x_test)

In [77]:
import tensorflow as tf

In [79]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(units=128, activation='relu',
                          input_dim=x_train_scaled.shape[1]),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

In [90]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
learning_rate = 0.01
metrics = ['accuracy']
momentum = 0.9

optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                    momentum=momentum,
                                    nesterov = True)

In [91]:
model.compile(loss=loss_fn, optimizer=optimizer, metrics=metrics)

In [94]:
model.fit(x_train_scaled,
          y_train,
          epochs=50,
          batch_size=8,
          validation_split=0.1
          )

Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8609 - loss: 0.2913 - val_accuracy: 0.9028 - val_loss: 0.2909
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8797 - loss: 0.2931 - val_accuracy: 0.9167 - val_loss: 0.2904
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8609 - loss: 0.2990 - val_accuracy: 0.8889 - val_loss: 0.3232
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8625 - loss: 0.2891 - val_accuracy: 0.9028 - val_loss: 0.2751
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8609 - loss: 0.2933 - val_accuracy: 0.8750 - val_loss: 0.3153
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8687 - loss: 0.2898 - val_accuracy: 0.8611 - val_loss: 0.3100
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8609 - loss: 0.2919 - val_accuracy: 0.8750 - val_loss: 0.3135
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8672 - loss: 0.2896 - val_accuracy: 0.8750 - val_loss:

In [95]:
loss, accuracy = model.evaluate(x_test_scaled,y_test)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8268 - loss: 0.5059
